In [1]:
import os

os.environ["MUJOCO_GL"] = "egl"

In [2]:
import argparse
import pathlib
import sys

import numpy as np
import torch
#from omegaconf import OmegaConf

import sys

sys.path.append("/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch")
import dreamer as dreamer_main
import models as dreamer_models
import tools as dreamer_tools


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
# Real Dreamer obs_batch from DMC cartpole_swingup (vision + proprio)
import numpy as np
import torch

from envs import dmc
import envs.wrappers as wrappers

T = 10  # rollout length

# Create env (matches Dreamer DMC defaults)
env = dmc.DeepMindControl("cartpole_swingup",
                          action_repeat=1,
                          size=(64, 64),
                          seed=0)
env = wrappers.NormalizeActions(env)

obs_list = []
action_list = []
reward_list = [0.0]
discount_list = [1.0]

obs = env.reset()
obs_list.append(obs)

for _ in range(T - 1):
    action = env.action_space.sample()
    next_obs, reward, done, info = env.step(action)
    action_list.append(action)
    reward_list.append(reward)
    discount_list.append(float(info.get("discount", 1.0)))
    obs_list.append(next_obs)
    if done:
        break

# Build obs_batch with batch dimension
obs_batch = {}
for key in obs_list[0].keys():
    obs_batch[key] = np.stack([o[key] for o in obs_list], axis=0)[None, ...]

# Actions: zero for t=0, then executed actions
action_dim = env.action_space.shape[0]
actions = np.zeros((len(obs_list), action_dim), dtype=np.float32)
if action_list:
    actions[1:1 + len(action_list)] = np.stack(action_list, axis=0)
obs_batch["action"] = actions[None, ...]

# Rewards/discounts aligned with obs_list
obs_batch["reward"] = np.array(reward_list, dtype=np.float32)[None, ...]
obs_batch["discount"] = np.array(discount_list, dtype=np.float32)[None, ...]

# Use the same actions downstream for rollouts
actions = obs_batch["action"]


libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card3: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card2: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card0: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card3: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card2: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card0: Permission denied

/home/hgf_hmgu/hgf_gib4562/miniconda3/envs/dreamerv3/lib/python3.11/site

In [4]:
obs_space = env.observation_space

In [5]:
# obs_space = _build_obs_space(obs)

In [6]:
import ruamel.yaml as yaml


def _parse_dreamer_config(configs_path, config_names, overrides):
    configs = yaml.safe_load(pathlib.Path(configs_path).read_text())

    def recursive_update(base, update):
        for key, value in update.items():
            if isinstance(value, dict) and key in base:
                recursive_update(base[key], value)
            else:
                base[key] = value

    name_list = ["defaults", *config_names] if config_names else ["defaults"]
    defaults = {}
    for name in name_list:
        recursive_update(defaults, configs[name])

    overrides_map = {}
    for item in overrides:
        if item.startswith("--"):
            item = item[2:]
        if "=" in item:
            key, value = item.split("=", 1)
            if key not in defaults:
                raise KeyError(f"Unknown config key '{key}' in override.")
            cast = dreamer_tools.args_type(defaults[key])
            overrides_map[key] = cast(value)
        else:
            if item not in defaults:
                raise KeyError(f"Unknown config key '{item}' in override.")
            overrides_map[item] = True

    merged = {**defaults, **overrides_map}
    return argparse.Namespace(**merged)


def rollout_features(wm, obs_batch, actions, deterministic):
    data = wm.preprocess(obs_batch)
    embed = wm.encoder(data)
    post, _ = wm.dynamics.observe(embed, data["action"], data["is_first"])
    state = {k: v[:, 0] for k, v in post.items()}

    feats = [wm.dynamics.get_feat(state)]
    for t in range(actions.shape[1]):
        state = wm.dynamics.img_step(state,
                                     actions[:, t],
                                     sample=not deterministic)
        feats.append(wm.dynamics.get_feat(state))
    feats = torch.stack(feats, dim=1)
    return feats

In [7]:
cfg = _parse_dreamer_config(
    "/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/configs.yaml",
    ["dmc_proprio"],
    ["task=dmc_cartpole_swingup"],
)

In [8]:
ckpt_path = '/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/logdir/dmc_cartpole_swingup_srun/latest.pt'
cfg.num_actions = 1
device = 'cuda:0'

wm = dreamer_models.WorldModel(obs_space, None, step=0, config=cfg).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
agent_state = ckpt.get("agent_state_dict", ckpt)
wm_state = {
    k[len("_wm."):]: v for k, v in agent_state.items() if k.startswith("_wm.")
}
wm.load_state_dict(wm_state, strict=False)
wm.eval()

Encoder CNN shapes: {}
Encoder MLP shapes: {'position': (3,), 'velocity': (2,)}
Decoder CNN shapes: {}
Decoder MLP shapes: {'position': (3,), 'velocity': (2,)}


/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/tools.py:747: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/scratch/slurm_tmpdir/job_1601401/ipykernel_28493/686650303.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you s

Optimizer model_opt has 16428293 variables.


WorldModel(
  (encoder): MultiEncoder(
    (_mlp): MLP(
      (layers): Sequential(
        (Encoder_linear0): Linear(in_features=5, out_features=1024, bias=False)
        (Encoder_norm0): LayerNorm((1024,), eps=0.001, elementwise_affine=True)
        (Encoder_act0): SiLU()
        (Encoder_linear1): Linear(in_features=1024, out_features=1024, bias=False)
        (Encoder_norm1): LayerNorm((1024,), eps=0.001, elementwise_affine=True)
        (Encoder_act1): SiLU()
        (Encoder_linear2): Linear(in_features=1024, out_features=1024, bias=False)
        (Encoder_norm2): LayerNorm((1024,), eps=0.001, elementwise_affine=True)
        (Encoder_act2): SiLU()
        (Encoder_linear3): Linear(in_features=1024, out_features=1024, bias=False)
        (Encoder_norm3): LayerNorm((1024,), eps=0.001, elementwise_affine=True)
        (Encoder_act3): SiLU()
        (Encoder_linear4): Linear(in_features=1024, out_features=1024, bias=False)
        (Encoder_norm4): LayerNorm((1024,), eps=0.001, eleme

In [9]:
actions = torch.tensor(actions, device=device, dtype=torch.float32)
feats = rollout_features(wm, obs_batch, actions, True)

In [10]:
feats.shape

torch.Size([1, 11, 1536])

In [11]:
print("dyn_discrete:", cfg.dyn_discrete)
print("dyn_stoch:", cfg.dyn_stoch, "dyn_deter:", cfg.dyn_deter)

if cfg.dyn_discrete:
    feat_dim = cfg.dyn_deter + cfg.dyn_stoch * cfg.dyn_discrete
else:
    feat_dim = cfg.dyn_deter + cfg.dyn_stoch

print("expected feat dim:", feat_dim)

dyn_discrete: 32
dyn_stoch: 32 dyn_deter: 512
expected feat dim: 1536
